In [1]:
import pandas as pd

In [10]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"
columns = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'tip_amount', 'total_amount']
df = pd.read_parquet(url, columns=columns)

In [ ]:
# 4

In [3]:

# ensure datetime
df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'])

# set pickup as index (for easier resampling)
df.set_index('lpep_pickup_datetime', inplace=True)

# group by 5-minute windows and PULocationID, count trips
trips_per_window = (
    df.groupby([pd.Grouper(freq='5min'), 'PULocationID'])
      .size()
      .reset_index(name='num_trips')
)

# find the PULocationID with the most trips in a single 5-min window
top_pu = trips_per_window.sort_values('num_trips', ascending=False).head(3)
print(top_pu)

      lpep_pickup_datetime  PULocationID  num_trips
23500  2025-10-22 08:40:00            74         15
21829  2025-10-20 16:30:00            74         14
15690  2025-10-15 08:45:00            74         13


In [ ]:
# 5

In [8]:
# ensure sorted by pickup time
df = df.sort_values(['PULocationID', 'lpep_pickup_datetime'])

# compute time difference between consecutive trips for the same PULocationID
df['prev_time'] = df.groupby('PULocationID')['lpep_pickup_datetime'].shift()
df['gap'] = (df['lpep_pickup_datetime'] - df['prev_time']).dt.total_seconds() / 60  # in minutes

# start a new session if gap > 5 min or first row
df['new_session'] = (df['gap'] > 5) | (df['gap'].isna())

# assign session IDs
df['session_id'] = df.groupby('PULocationID')['new_session'].cumsum()

# count trips per session
session_counts = df.groupby(['PULocationID', 'session_id']).size().reset_index(name='num_trips')

# find the session with most trips
longest_session = session_counts.sort_values('num_trips', ascending=False).head(1)
print(longest_session)

      PULocationID  session_id  num_trips
8398            74         450         82


In [13]:
# ensure datetime and set index
df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'])
df.set_index('lpep_pickup_datetime', inplace=True)

# resample by 1-hour windows (use lowercase 'h')
hourly_tips = df['tip_amount'].resample('1h').sum().reset_index()

# find the hour with the highest total tip
top_hour = hourly_tips.sort_values('tip_amount', ascending=False).head(1)
print(top_hour)

    lpep_pickup_datetime  tip_amount
524  2025-10-16 18:00:00      524.96
